# Tabla de predictores y puertas P2 y P3 — modelo perinatal NCHS

Etapas del framework: **E7** (predictores), puerta **P2** (fuga) y puerta **P3** (tamaño de muestra). Construye la tabla contrato `data/contract/perinatal_nchs_v1/year_<año>.parquet` (no se versiona) con los 24 parámetros de la lista de la auditoría de conteos, aplicando las decisiones D3 (el desconocido es faltante, sin indicadores) y D4 (sin variables que dependen de la duración del embarazo). Salida agregada: `data/outputs_03_predictors_g2_g3.txt`. Versión en inglés: `03_predictors_g2_g3.ipynb` (mismo código). Tarda unos 4 minutos la primera vez.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.perinatal_builder import YEARS
from libraries.nchs_reader import OUT_DIR

import os
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_03_predictors_g2_g3.txt"
DEV_YEARS = [y for y in YEARS if y <= 2021]          # temporal test: 2022-2023 (criterion C4)
SEED = 20261121
lines = []

def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

def yn(s):
    """Y -> 1, N -> 0, anything else (U, blank) -> missing (decision D3)."""
    s = s.astype("string")
    y = s.eq("Y").fillna(False).to_numpy(bool)
    n = s.eq("N").fillna(False).to_numpy(bool)
    return pd.Series(np.where(y, 1.0, np.where(n, 0.0, np.nan)), index=s.index)

def coded(s, unknown):
    s = s.astype("Float64").astype(float)
    return s.where(~s.isin(unknown))

def make_predictors(d):
    """Predictor table of the outcome card (count audit, section 5; decisions D3 and D4).
    Unknown codes become NaN; no missingness indicators. Reference levels: White,
    high school or less, no earlier living birth, first child, care in the first trimester."""
    X = pd.DataFrame(index=d.index)
    X["age"] = coded(d["MAGER"], [])
    race = coded(d["MRACEHISP"], [8])
    for name, codes in [("race_black", [2]), ("race_hispanic", [7]), ("race_asian", [4]), ("race_other", [3, 5, 6])]:
        X[name] = np.where(race.isna(), np.nan, race.isin(codes).astype(float))
    edu = coded(d["MEDUC"], [9])
    X["educ_some_college"] = np.where(edu.isna(), np.nan, edu.isin([4, 5]).astype(float))
    X["educ_bachelor_plus"] = np.where(edu.isna(), np.nan, edu.isin([6, 7, 8]).astype(float))
    nat = coded(d["MBSTATE_REC"], [3])
    X["born_outside_us"] = np.where(nat.isna(), np.nan, (nat == 2).astype(float))
    X["wic"] = yn(d["WIC"])
    live = coded(d["PRIORLIVE"], [99])
    X["prior_living_1"] = np.where(live.isna(), np.nan, (live == 1).astype(float))
    X["prior_living_2plus"] = np.where(live.isna(), np.nan, (live >= 2).astype(float))
    dead = coded(d["PRIORDEAD"], [99])
    X["prior_child_died"] = np.where(dead.isna(), np.nan, (dead >= 1).astype(float))
    ilb = coded(d["ILLB_R"], [999])
    ilb = ilb.where(~ilb.between(0, 3))                 # 000-003 = plural delivery code, not an interval
    first = ilb == 888
    for name, lo, hi in [("interval_lt18", 4, 17), ("interval_18_59", 18, 59), ("interval_60plus", 60, 300)]:
        X[name] = np.where(ilb.isna(), np.nan, (~first & ilb.between(lo, hi)).astype(float))
    X["prior_cesarean"] = yn(d["RF_CESAR"])
    bmi = d["BMI"].astype("Float64").astype(float)
    X["bmi"] = bmi.where((bmi < 99) & bmi.between(12, 80))
    cig = coded(d["CIG_0"], [99])
    X["smoked_before_pregnancy"] = np.where(cig.isna(), np.nan, (cig >= 1).astype(float))
    pc = coded(d["PRECARE"], [99])
    X["care_after_first_trimester"] = np.where(pc.isna(), np.nan, pc.between(4, 10).astype(float))
    X["no_prenatal_care"] = np.where(pc.isna(), np.nan, (pc == 0).astype(float))
    X["pregestational_diabetes"] = yn(d["RF_PDIAB"])
    X["chronic_hypertension"] = yn(d["RF_PHYPE"])
    X["infertility_treatment"] = yn(d["RF_INFT"])
    X["multiple_pregnancy"] = coded(d["DPLURAL"], []).gt(1).astype(float)
    return X.astype("float32")

PREDICTORS = list(make_predictors(pd.read_parquet(OUT_DIR / "perinatal_2023.parquet").head(5)).columns)
print(len(PREDICTORS), "parameters:", PREDICTORS)

24 parameters: ['age', 'race_black', 'race_hispanic', 'race_asian', 'race_other', 'educ_some_college', 'educ_bachelor_plus', 'born_outside_us', 'wic', 'prior_living_1', 'prior_living_2plus', 'prior_child_died', 'interval_lt18', 'interval_18_59', 'interval_60plus', 'prior_cesarean', 'bmi', 'smoked_before_pregnancy', 'care_after_first_trimester', 'no_prenatal_care', 'pregestational_diabetes', 'chronic_hypertension', 'infertility_treatment', 'multiple_pregnancy']


## 1. Tabla contrato y cobertura de los predictores (toda la población)

In [2]:
# Contract table (one file per delivery year) and predictor coverage in the whole population
REBUILD = False
KEEP = ["unit_id", "group_id", "year", "component", "ga_weeks", "label_main", "label_fetal",
        "label_early_neonatal", "estofd"]
CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
for y in YEARS:
    final = CONTRACT_DIR / f"year_{y}.parquet"
    if final.exists() and not REBUILD:
        continue
    d = pd.read_parquet(OUT_DIR / f"perinatal_{y}.parquet")
    d = d[d["in_population"]].reset_index(drop=True)
    X = make_predictors(d)
    c = pd.concat([d[KEEP].rename(columns={"year": "period"}), X], axis=1)
    c["label_ge28"] = c["label_main"] & (c["ga_weeks"] >= 28)
    c["n_unknown"] = X.isna().sum(axis=1).astype("int8")      # for the leakage check only, never a predictor
    tmp = final.with_suffix(".parquet.tmp")
    c.to_parquet(tmp, index=False); tmp.replace(final)
    print(f"year {y}: {len(c)} products written")
    del d, X, c

miss, n_total = {}, 0
for y in YEARS:
    X = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=PREDICTORS)
    n_total += len(X)
    for p in PREDICTORS:
        miss[p] = miss.get(p, 0) + int(X[p].isna().sum())
cov = pd.DataFrame({"pct_missing": {p: round(100 * miss[p] / n_total, 2) for p in PREDICTORS}})
show(f"Q-P1 Contract table: {n_total} products, {len(PREDICTORS)} parameters; % missing per predictor", cov)


== Q-P1 Contract table: 29963276 products, 24 parameters; % missing per predictor
                            pct_missing
age                                0.00
race_black                         0.92
race_hispanic                      0.92
race_asian                         0.92
race_other                         0.92
educ_some_college                  1.49
educ_bachelor_plus                 1.49
born_outside_us                    0.23
wic                                1.19
prior_living_1                     0.17
prior_living_2plus                 0.17
prior_child_died                   0.28
interval_lt18                      4.50
interval_18_59                     4.50
interval_60plus                    4.50
prior_cesarean                     0.13
bmi                                2.28
smoked_before_pregnancy            0.48
care_after_first_trimester         2.31
no_prenatal_care                   2.31
pregestational_diabetes            0.13
chronic_hypertension               0.

## 2. Puerta P2: verificaciones de diseño y prueba de fuga del número de desconocidos

In [3]:
# Gate G2 (leakage)
EXCLUDED = ["ga_weeks", "birthweight", "estofd", "component", "label_main", "label_fetal",
            "label_early_neonatal", "label_ge28", "n_unknown", "RF_GDIAB", "RF_GHYPE", "RF_EHYPE",
            "CIG_REC", "PREVIS", "WTGAIN", "APGAR5", "SEX"]
checks = {
    "no delivery, product, outcome or excluded field among the predictors": not set(PREDICTORS) & set(EXCLUDED),
    "no missingness indicator among the predictors": not any("missing" in p or "unknown" in p for p in PREDICTORS),
    "unit of analysis is the product and the group is the unit itself (no mother identifier)": True,
}
show("Gate G2: design checks", pd.Series(checks))
assert all(checks.values())

# Leakage check of decision D3: how well does the NUMBER of unknown fields alone separate events?
tab = None
for y in DEV_YEARS:
    c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=["n_unknown", "label_main"])
    t = pd.crosstab(c["n_unknown"], c["label_main"])
    tab = t if tab is None else tab.add(t, fill_value=0)
tab = tab.reindex(columns=[False, True], fill_value=0).astype(np.int64)

def auroc_from_counts(neg, pos):
    """Exact AUROC (ties count one half) of an ordinal score from counts per score value."""
    neg, pos = np.asarray(neg, float), np.asarray(pos, float)
    below = np.concatenate([[0], np.cumsum(neg)[:-1]])
    return float(((below + neg / 2) * pos).sum() / (neg.sum() * pos.sum()))

auc_unknown = auroc_from_counts(tab[False], tab[True])
show("G2 leakage check: products by number of unknown predictor fields (development years)",
     pd.DataFrame({"products": tab.sum(axis=1), "pct_products": (100 * tab.sum(axis=1) / tab.values.sum()).round(2)}))
show("G2 leakage check: AUROC of the number of unknown fields alone", round(auc_unknown, 4))


== Gate G2: design checks
no delivery, product, outcome or excluded field among the predictors                       True
no missingness indicator among the predictors                                              True
unit of analysis is the product and the group is the unit itself (no mother identifier)    True

== G2 leakage check: products by number of unknown predictor fields (development years)
           products  pct_products
n_unknown                        
0          20198572         89.05
1            591646          2.61
2            509795          2.25
3            949750          4.19
4            151598          0.67
5             80132          0.35
6            125366          0.55
7             23857          0.11
8             19607          0.09
9             12901          0.06
10             4667          0.02
11             3523          0.02
12             2288          0.01
13             1657          0.01
14             1050          0.00
15              94

## 3. Puerta P3: criterios de Riley con el R² de Cox-Snell observado (años de desarrollo)

In [4]:
# Gate G3 (sample size): Riley et al. with the observed Cox-Snell R2 of a cross-validated
# logistic regression. Development years only. All events and a random fraction of non-events
# are used, with inverse-probability weights, so the log-likelihood estimates that of the full data.
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

FRACTION = 0.03            # share of non-events sampled
rng = np.random.default_rng(SEED)
parts, n_dev, e_dev = [], 0, 0
for y in DEV_YEARS:
    c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=PREDICTORS + ["label_main"])
    n_dev += len(c); e_dev += int(c["label_main"].sum())
    keep = c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)
    parts.append(c[keep])
s = pd.concat(parts, ignore_index=True); del parts
y_s = s.pop("label_main").to_numpy(bool)
w_s = np.where(y_s, 1.0, 1.0 / FRACTION)
P = len(PREDICTORS)

oof = np.zeros(len(s))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(s, y_s):
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(C=1e6, max_iter=1000))
    m.fit(s.iloc[tr], y_s[tr], logisticregression__sample_weight=w_s[tr])
    oof[te] = m.predict_proba(s.iloc[te])[:, 1]

p = np.clip(oof, 1e-12, 1 - 1e-12)
L1 = float(np.sum(w_s * (y_s * np.log(p) + (~y_s) * np.log(1 - p))))
n = float(w_s.sum()); phi = e_dev / n_dev
L0 = n * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
R2 = 1 - np.exp(2 * (L0 - L1) / n)
R2max = 1 - np.exp(2 * L0 / n)

S = 0.9
n1 = P / ((S - 1) * np.log(1 - R2 / S))
S2 = R2 / (R2 + 0.05 * R2max)
n2 = P / ((S2 - 1) * np.log(1 - R2 / S2))
n3 = (1.96 / 0.05) ** 2 * phi * (1 - phi)
n_req = int(np.ceil(max(n1, n2, n3)))
Smax = max(S, S2)
p_max = int(np.floor(n_dev * (Smax - 1) * np.log(1 - R2 / Smax)))
gate = "PASSED" if n_dev >= n_req else "NOT PASSED"
g3 = pd.Series({"development products": n_dev, "development events": e_dev, "prevalence": round(phi, 5),
                "candidate parameters": P, "events per parameter": round(e_dev / P, 1),
                "sample used (all events + 3% of non-events)": len(s),
                "Cox-Snell R2 observed (cross-validated)": round(R2, 5), "Cox-Snell R2 max": round(R2max, 5),
                "% of maximum": round(100 * R2 / R2max, 1),
                "n required (shrinkage)": int(np.ceil(n1)), "n required (optimism)": int(np.ceil(n2)),
                "n required (overall risk)": int(np.ceil(n3)), "n required": n_req,
                "events required": int(np.ceil(n_req * phi)), "parameters supported": p_max, "gate G3": gate})
show("Gate G3: sample size (Riley et al.)", g3)
show("Primary learner rule (events per parameter >= 20: best base learner of a preliminary run)",
     "events per parameter = %.1f" % (e_dev / P))

OUT_FILE.write_text("Predictor table and gates G2 and G3 of the perinatal model (aggregate output only)\n"
                    + "\n".join(lines) + "\n", encoding="utf-8")


== Gate G3: sample size (Riley et al.)
development products                           22681046
development events                               149057
prevalence                                      0.00657
candidate parameters                                 24
events per parameter                             6210.7
sample used (all events + 3% of non-events)      825892
Cox-Snell R2 observed (cross-validated)          0.0058
Cox-Snell R2 max                                 0.0761
% of maximum                                        7.6
n required (shrinkage)                            37147
n required (optimism)                              6278
n required (overall risk)                            11
n required                                        37147
events required                                     245
parameters supported                              14654
gate G3                                          PASSED

== Primary learner rule (events per parameter >= 20: best base 

3552